# LSTM

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_recurrent-modern/lstm.ipynb` · [Lección original](https://d2l.ai/chapter_recurrent-modern/lstm.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Memoria a largo y corto plazo (LSTM)
<a id="sec_lstm"></a>

Poco después de entrenar las primeras RNN de tipo Elman mediante retropropagación [elman1990finding](https://d2l.ai/chapter_references/zreferences.html), se hicieron evidentes las dificultades para aprender dependencias lejanas: los gradientes podían desvanecerse o explotar. Bengio y Hochreiter analizaron este problema [bengio1994learning,Hochreiter.Bengio.Frasconi.ea.2001](https://d2l.ai/chapter_references/zreferences.html); Hochreiter ya lo había descrito en su tesis de 1991, escrita en alemán. El recorte de gradientes ayuda frente a valores excesivos, pero recuperar señales que se desvanecen requiere otros mecanismos.

Una respuesta especialmente influyente fue LSTM [Hochreiter.Schmidhuber.1997](https://d2l.ai/chapter_references/zreferences.html). En lugar de una unidad recurrente simple, introduce una *celda de memoria* con estado interno y conexiones multiplicativas controladas por compuertas. La ruta aditiva de la celda facilita transportar información y gradientes durante más pasos. No significa que una LSTM sea inmune a cualquier problema de gradientes: el comportamiento depende de las compuertas y del entrenamiento.

El nombre distingue tres escalas. Los pesos almacenan conocimiento que cambia lentamente durante el entrenamiento; las activaciones recurrentes forman una memoria de corto plazo; la celda proporciona un mecanismo para conservar algunas de esas activaciones durante más tiempo. Estudiaremos cómo se decide qué escribir, qué conservar y qué exponer como salida.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Celda de memoria con compuertas
Cada célula de memoria está equipada con un *estado interno* y un número de puertas multiplicativas que determinan si (i) una entrada dada debe impactar el estado interno (la *compuerta de entrada*), (ii) el estado interno debe ser lanzado a $0$ (la *compuerta de olvido*), y (iii) el estado interno de una neurona dada debe ser permitido impactar la salida de la célula (la compuerta de salida*).

### Estado oculto con compuertas
La distinción clave entre los RNN convencionales y los LSTMs es que estos últimos apoyan la fijación del estado oculto. Esto significa que tenemos mecanismos dedicados para cuando un estado oculto debe ser actualizado y también para cuando debe ser restaurado. Estos mecanismos se aprenden y abordan las preocupaciones enumeradas anteriormente. Por ejemplo, si el primer símbolo es de gran importancia aprenderemos a no actualizar el estado oculto después de la primera observación. Igualmente, aprenderemos a saltar observaciones temporales irrelevantes. Por último, aprenderemos a restablecer el estado latente cuando sea necesario. Discutimos esto en detalle a continuación.

### Puerta de entrada, compuerta de olvido y compuerta de salida
La entrada de datos en las puertas LSTM es la entrada en el paso de tiempo actual y el estado oculto del paso de tiempo anterior, como se ilustra en [Referencia fig_lstm_0](https://d2l.ai/chapter_recurrent-modern/lstm.html#fig-lstm-0). Tres capas totalmente conectadas con funciones de activación sigmoide calculan los valores de las puertas de entrada, olvido y salida. Como resultado de la activación sigmoide, todos los valores de las tres puertas están en el rango de $(0, 1)$. Además, necesitamos un *nodo de entrada*, normalmente calculado con una función de activación *tanh*. Intuitivamente, la *compuerta de entrada* determina cuánto valor del nodo de entrada debe añadirse al estado interno de la celda de memoria actual. La *compuerta de olvido* determina si mantener el valor actual de la memoria o limpiarla. Y la *compuerta de salida* determina si la celda de memoria debe influir en la salida en el paso de tiempo actual.

![Cálculo de las compuertas de entrada, olvido y salida en una LSTM.](../recursos/originales/lstm-0.svg)
<a id="fig_lstm_0"></a>

Matemáticamente, supongamos que hay $h$ unidades ocultas, el tamaño del lote es $n$, y el número de entradas es $d$. Por lo tanto, la entrada es $\mathbf{X}_t \in \mathbb{R}^{n \times d}$ y el estado oculto del paso de tiempo anterior es $\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$. Correspondientemente, las puertas en el paso de tiempo $t$ se definen como sigue: la compuerta de entrada es $\mathbf{I}_t \in \mathbb{R}^{n \times h}$, la compuerta de olvido es $\mathbf{F}_t \in \mathbb{R}^{n \times h}$, y la compuerta de salida es $\mathbf{O}_t \in \mathbb{R}^{n \times h}$. Se calculan como sigue:

$$
\begin{aligned}
\mathbf{I}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xi}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hi}} + \mathbf{b}_\textrm{i}),\\
\mathbf{F}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xf}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hf}} + \mathbf{b}_\textrm{f}),\\
\mathbf{O}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{\textrm{xo}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{ho}} + \mathbf{b}_\textrm{o}),
\end{aligned}
$$

donde $\mathbf{W}_{\textrm{xi}}, \mathbf{W}_{\textrm{xf}}, \mathbf{W}_{\textrm{xo}} \in \mathbb{R}^{d \times h}$ y $\mathbf{W}_{\textrm{hi}}, \mathbf{W}_{\textrm{hf}}, \mathbf{W}_{\textrm{ho}} \in \mathbb{R}^{h \times h}$ son parámetros de peso y $\mathbf{b}_\textrm{i}, \mathbf{b}_\textrm{f}, \mathbf{b}_\textrm{o} \in \mathbb{R}^{1 \times h}$ son parámetros de sesgo. Tenga en cuenta que la transmisión (ver [Referencia subsec_broadcasting](https://d2l.ai/chapter_preliminaries/ndarray.html#subsec-broadcasting)) se activa durante la suma. Utilizamos funciones sigmoide (como se introdujo en [Referencia sec_mlp](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#sec-mlp)) para mapear los valores de entrada al intervalo $(0, 1)$.

### Nodo de entrada
A continuación diseñamos la celda de memoria. Ya que no hemos especificado la acción de las diferentes puertas todavía, primero introducimos el *nodo de entrada* $\tilde{\mathbf{C}}_t \in \mathbb{R}^{n \times h}$. Su cálculo es similar a la de las tres puertas descritas anteriormente, pero utiliza una función $\tanh$ con un rango de valor para $(-1, 1)$ como función de activación. Esto conduce a la siguiente ecuación en el paso de tiempo $t$:

$$\tilde{\mathbf{C}}_t = \textrm{tanh}(\mathbf{X}_t \mathbf{W}_{\textrm{xc}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hc}} + \mathbf{b}_\textrm{c}),$$

donde $\mathbf{W}_{\textrm{xc}} \in \mathbb{R}^{d \times h}$ y $\mathbf{W}_{\textrm{hc}} \in \mathbb{R}^{h \times h}$ son parámetros de peso y $\mathbf{b}_\textrm{c} \in \mathbb{R}^{1 \times h}$ es un parámetro de sesgo.

En [Referencia fig_lstm_1](https://d2l.ai/chapter_recurrent-modern/lstm.html#fig-lstm-1) se muestra una ilustración rápida del nodo de entrada.

![Cálculo del nodo de entrada en una LSTM.](../recursos/originales/lstm-1.svg)
<a id="fig_lstm_1"></a>

### Estado interno de la célula de memoria
En LSTMs, la compuerta de entrada $\mathbf{I}_t$ rige cuánto tomamos en cuenta los nuevos datos a través de $\tilde{\mathbf{C}}_t$ y la compuerta de olvido $\mathbf{F}_t$ se dirige a cuánto del viejo estado interno de la celda $\mathbf{C}_{t-1} \in \mathbb{R}^{n \times h}$ que retenemos. Usando el operador de producto Hadamard (elementwise) $\odot$ llegamos a la siguiente ecuación de actualización:

$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t.$$

Si la puerta del olvido es siempre 1 y la compuerta de entrada es siempre 0, el estado interno de la celda de memoria $\mathbf{C}_{t-1}$ permanecerá constante para siempre, pasando sin cambios a cada paso de tiempo posterior. Sin embargo, las puertas de entrada y las puertas del olvido dan al modelo la flexibilidad de poder aprender cuándo mantener este valor sin cambios y cuándo perturbarlo en respuesta a las entradas posteriores. En la práctica, este diseño alivia el problema de desvanecimiento de gradientes, resultando en modelos que son mucho más fáciles de entrenar, especialmente cuando se enfrentan a conjuntos de datos con largas longitudes de secuencia.

Llegamos así al diagrama de flujo en [Referencia fig_lstm_2](https://d2l.ai/chapter_recurrent-modern/lstm.html#fig-lstm-2).

![Cálculo del estado interno de la celda de memoria en una LSTM.](../recursos/originales/lstm-2.svg)

<a id="fig_lstm_2"></a>

### Estado oculto
Por último, necesitamos definir cómo calcular la salida de la celda de memoria, es decir, el estado oculto $\mathbf{H}_t \in \mathbb{R}^{n \times h}$, como se ve por otras capas. Aquí es donde entra en juego la compuerta de salida. En LSTMs, primero aplicamos $\tanh$ al estado interno de la celda de memoria y luego aplicamos otra multiplicación en sentido punto, esta vez con la compuerta de salida. Esto asegura que los valores de $\mathbf{H}_t$ estén siempre en el intervalo $(-1, 1)$:

$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t).$$

Cada vez que la compuerta de salida está cerca de 1, permitimos que el estado interno de la celda de memoria impacte las capas subsiguientes desinhibidas, mientras que para los valores de la compuerta de salida cercanos a 0, evitamos que la memoria actual impacte otras capas de la red en el paso actual del tiempo. Tenga en cuenta que una celda de memoria puede acumular información a través de muchos pasos de tiempo sin impactar el resto de la red (siempre que la compuerta de salida tome valores cercanos a 0), y luego impactar repentinamente la red en un paso de tiempo posterior tan pronto como la compuerta de salida gire de valores cercanos a 0 a valores cercanos a 1. [Referencia fig_lstm_3](https://d2l.ai/chapter_recurrent-modern/lstm.html#fig-lstm-3) tiene una ilustración gráfica del flujo de datos.

![Cálculo del estado oculto en una LSTM.](../recursos/originales/lstm-3.svg)
<a id="fig_lstm_3"></a>

## Implementación desde cero
Ahora vamos a implementar un LSTM desde cero. Al igual que los experimentos en [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch), primero cargamos *The Time Machine* dataset.

### Inicialización de los parámetros del modelo

A continuación, tenemos que definir e inicializar los parámetros del modelo. Como antes, el hiperparametro `num_hiddens` dicta el número de unidades ocultas. Iniciamos pesos siguiendo una distribución gaussiana con 0.01 desviación estándar, y fijamos los sesgos en 0.


In [ ]:
class LSTMScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()

        init_weight = lambda *shape: nn.Parameter(torch.randn(*shape) * sigma)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          nn.Parameter(torch.zeros(num_hiddens)))
        self.W_xi, self.W_hi, self.b_i = triple()  # Puerta de entrada
        self.W_xf, self.W_hf, self.b_f = triple()  # Olvida la puerta.
        self.W_xo, self.W_ho, self.b_o = triple()  # Puerta de salida
        self.W_xc, self.W_hc, self.b_c = triple()  # Nodo de entrada

**El modelo actual** se define como descrito anteriormente, que consiste en tres puertas y un nodo de entrada. Tenga en cuenta que sólo el estado oculto se pasa a la capa de salida.


In [ ]:
@d2l.add_to_class(LSTMScratch)
def forward(self, inputs, H_C=None):
    if H_C is None:
        # Estado inicial con forma: (batch_size, num_hiddens)
        H = torch.zeros((inputs.shape[1], self.num_hiddens),
                      device=inputs.device)
        C = torch.zeros((inputs.shape[1], self.num_hiddens),
                      device=inputs.device)
    else:
        H, C = H_C
    outputs = []
    for X in inputs:
        I = torch.sigmoid(torch.matmul(X, self.W_xi) +
                        torch.matmul(H, self.W_hi) + self.b_i)
        F = torch.sigmoid(torch.matmul(X, self.W_xf) +
                        torch.matmul(H, self.W_hf) + self.b_f)
        O = torch.sigmoid(torch.matmul(X, self.W_xo) +
                        torch.matmul(H, self.W_ho) + self.b_o)
        C_tilde = torch.tanh(torch.matmul(X, self.W_xc) +
                           torch.matmul(H, self.W_hc) + self.b_c)
        C = F * C + I * C_tilde
        H = O * torch.tanh(C)
        outputs.append(H)
    return outputs, (H, C)

### Entrenamiento** y predicción
Entrenemos un modelo LSTM presentando la clase `RNNLMScratch` de [Referencia sec_rnn-scratch](https://d2l.ai/chapter_recurrent-neural-networks/rnn-scratch.html#sec-rnn-scratch).


### Nota docente de Hespérides

Una RNN reutiliza parámetros a lo largo del tiempo. En BPTT, las contribuciones recorren productos de Jacobianos, que pueden reducirse o crecer repetidamente. LSTM añade un estado de celda y compuertas para modular retención, escritura y lectura; GRU ofrece una alternativa más compacta. Las compuertas no garantizan recordar cualquier secuencia. Observa la longitud de truncamiento, la norma del gradiente y qué estado se conserva entre fragmentos.

Vínculo con los apuntes: sesión 5, «LSTM».


In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)
lstm = LSTMScratch(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLMScratch(lstm, vocab_size=len(data.vocab), lr=4)
trainer = d2l.Trainer(max_epochs=50, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)

## Implementación concisa

Utilizando API de alto nivel, podemos instanciar directamente un modelo LSTM. Esto encapsula todos los detalles de configuración que hemos hecho explícitos anteriormente. El código es significativamente más rápido ya que utiliza operadores compilados en lugar de Python para muchos detalles que hemos explicado antes.


In [ ]:
class LSTM(d2l.RNN):
    def __init__(self, num_inputs, num_hiddens):
        d2l.Module.__init__(self)
        self.save_hyperparameters()
        self.rnn = nn.LSTM(num_inputs, num_hiddens)

    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [ ]:
lstm = LSTM(num_inputs=len(data.vocab), num_hiddens=32)
model = d2l.RNNLM(lstm, vocab_size=len(data.vocab), lr=4)
trainer.fit(model, data)

In [ ]:
model.predict('it has', 20, data.vocab, d2l.try_gpu())

Los LSTM son el modelo autorregresivo de variables latentes prototípicas con control de estado no trivial. Muchas de sus variantes se han propuesto a lo largo de los años, por ejemplo, múltiples capas, conexiones residuales, diferentes tipos de regularización. Sin embargo, entrenar LSTMs y otros modelos de secuencia (como GRUs) es bastante costoso debido a la dependencia de largo alcance de la secuencia. Más tarde nos encontraremos con modelos alternativos como Transformers que se pueden utilizar en algunos casos.

## Resumen
Mientras que los LSTM se publicaron en 1997, alcanzaron gran protagonismo con algunas victorias en concursos de predicción a mediados de los años 2000, y se convirtieron en los modelos dominantes para el aprendizaje de secuencias desde 2011 hasta el surgimiento de los modelos Transformer, a partir de 2017.Incluso los Tranformers deben algunas de sus ideas clave a las innovaciones de diseño de arquitectura introducidas por el LSTM.

Los LSTM tienen tres tipos de puertas: puertas de entrada, puertas de olvido y puertas de salida que controlan el flujo de información. La salida de capa oculta de LSTM incluye el estado oculto y el estado interno de la célula de memoria. Sólo el estado oculto se pasa a la capa de salida mientras que el estado interno de la célula de memoria permanece completamente interno.

## Ejercicios
1. Ajuste los hiperparametros y analice su influencia en el tiempo de ejecución, la perplejidad y la secuencia de salida.
1. ¿Cómo se necesita cambiar el modelo para generar palabras adecuadas en lugar de secuencias de caracteres?
1. Compare el costo computacional para GRUs, LSTMs y RNNs regulares para una dimensión oculta dada. Preste especial atención al costo de entrenamiento e inferencia.
1. Dado que la celda de memoria candidata garantiza que el rango de valor está entre $-1$ y $1$ mediante el uso de la función $\tanh$, ¿por qué el estado oculto necesita utilizar la función $\tanh$ de nuevo para asegurarse de que el rango de valor de salida está entre $-1$ y $1$?
1. Implementar un modelo LSTM para la predicción de series temporales en lugar de la predicción de secuencias de caracteres.


[Debate del original](https://discuss.d2l.ai/t/1057)
